# CP8305 Final Project
## Notebook 2 – Data Preprocessing

**Team members:** Christopher Indris; Mojtaba Mohammadi; Mohammad Al Batayeh  
**Date:** 2026-03-09

---

### Instructions
Run with a fresh 3.11 environment (I used a 3.11.11 venv "cp8305fp").
Ensure that you pip install ipykernel to run the notebook.

---

### Objectives
- Remove duplicates and handle missing values.
- Detect and treat outliers.
- Encode categorical variables.
- Engineer new features using domain knowledge.
- Scale numerical features.
- Save the processed dataset for modelling.

## 1. Setup

In [96]:
%pip install --upgrade pip
%pip install ucimlrepo # for loading the dataset
%pip install scikit-learn # for data splitting and function application
%pip install matplotlib # for plotting
%pip install seaborn # for statistical plotting

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [97]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import io
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import (
    load_raw_data,
    drop_duplicate_rows,
    handle_missing_values,
    remove_outliers_iqr,
    encode_categoricals,
    save_processed_data,
)
from src.feature_engineering import (
    scale_features,
    add_interaction_term,
    add_log_transform,
    add_binned_column,
)
from src.visualization import plot_missing_values, save_figure

sns.set_theme(style='whitegrid')
%matplotlib inline

## 2. Load Data

In [98]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
diabetes_130_us_hospitals_for_years_1999_2008 = fetch_ucirepo(id=296) 
  
# data (as pandas dataframes) 
X = diabetes_130_us_hospitals_for_years_1999_2008.data.features 
y = diabetes_130_us_hospitals_for_years_1999_2008.data.targets 
idXy = diabetes_130_us_hospitals_for_years_1999_2008.data.original

/Users/chrisindris/.pyenv/versions/3.11.11/envs/cp8305fp/lib/python3.11/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


### Mappings for encoded values

In [99]:
def read_multiple_tables_csv(filepath, table_delimiter=',\n'):
    """
    Reads a single CSV file containing multiple tables into a list of DataFrames.
    """
    with open(filepath, "r") as f:
        csv_content = f.read()
    
    # Split the content into a list of strings, one for each table
    csv_tables_list = csv_content.split(table_delimiter)
    
    # Convert each string segment into a DataFrame
    dataframe_list = [pd.read_csv(io.StringIO(table_data)) for table_data in csv_tables_list if table_data.strip()]
    
    return dataframe_list

In [100]:
admission_type_ids, discharge_disposition_ids, admission_source_id = read_multiple_tables_csv('../data/diabetes130/raw/IDS_mapping.csv')

## 3. Remove Duplicates

There are no duplicates to drop.

In [101]:
int(X.duplicated().sum()), int(idXy.duplicated().sum())

(0, 0)

## 4. Missing Values

### Ordinal Categorical

- weight
Since weight is ordinal and categorical (weight is numeric, and it was mapped to ranges of 25 lbs.), we could estimate it by taking the mode (a kind of analogue to the mean). \
However, there may be correlations between it and other variables; in particular, age (which is ordinal categorical from a numeric, mapped to ranges of 10 years). Age also has no missing values. Hence, we could try setting for each row where weight is missing the weight equal to the mode of the weight of all rows where the age is the same as that row.

In [102]:
modes = X['weight'].groupby(X['age']).agg(pd.Series.mode).reset_index(drop=False)
modes['weight'] = modes['weight'].apply(lambda x: x if isinstance(x, str) else x[0])
modes

,age,weight
0,[0-10),[0-25)
1,[10-20),[50-75)
2,[20-30),[50-75)
3,[30-40),[75-100)
4,[40-50),[75-100)
5,[50-60),[75-100)
6,[60-70),[75-100)
7,[70-80),[75-100)
8,[80-90),[50-75)
9,[90-100),[50-75)


In [103]:
modes.loc[modes['age'] == '[0-10)', 'weight'].values[0]

'[0-25)'

In [104]:
columns_with_missing_values = diabetes_130_us_hospitals_for_years_1999_2008.variables[diabetes_130_us_hospitals_for_years_1999_2008.variables.missing_values == 'yes'].name.tolist()

In [105]:
idXy[columns_with_missing_values].isna().sum()

race                  2273
weight               98569
payer_code           40256
medical_specialty    49949
diag_1                  21
diag_2                 358
diag_3                1423
dtype: int64

In [106]:
idXy['weight'].unique()

<StringArray>
[        nan,  '[75-100)',   '[50-75)',    '[0-25)', '[100-125)',   '[25-50)',
 '[125-150)', '[175-200)', '[150-175)',      '>200']
Length: 10, dtype: str

In [107]:
idXy['weight'].value_counts()

weight
[75-100)     1336
[50-75)       897
[100-125)     625
[125-150)     145
[25-50)        97
[0-25)         48
[150-175)      35
[175-200)      11
>200            3
Name: count, dtype: int64

In [108]:
def replace_missing_values(df, columns, replace_with):
    if isinstance(columns, tuple):
        columns, helper_columns = columns
    if replace_with == 'mode':
        for col in columns:
            df[col] = df[col].fillna(df[col].mode().values[0], inplace=False)
    elif replace_with == 'mode_based_on_helper_column':
        for col in columns:
            modes = df[col].groupby(df[helper_columns[0]]).agg(pd.Series.mode).reset_index(drop=False)
            modes[col] = modes[col].apply(lambda x: x if isinstance(x, str) else x[0])
            df[col] = df.apply(lambda row: modes.loc[modes[helper_columns[0]] == row[helper_columns[0]], col].values[0] if pd.isna(row[col]) else row[col], axis=1)
    return df

In [109]:
idXy = replace_missing_values(idXy, (['weight'], ['age']), 'mode_based_on_helper_column')
X = replace_missing_values(X, (['weight'], ['age']), 'mode_based_on_helper_column')

In [110]:
idXy['weight'].unique()

<StringArray>
[   '[0-25)',   '[50-75)',  '[75-100)', '[100-125)',   '[25-50)', '[125-150)',
 '[175-200)', '[150-175)',      '>200']
Length: 9, dtype: str

In [111]:
idXy['weight'].value_counts()

weight
[75-100)     78129
[50-75)      22517
[100-125)      625
[0-25)         204
[125-150)      145
[25-50)         97
[150-175)       35
[175-200)       11
>200             3
Name: count, dtype: int64

### Approximately Ordinal Categorical (admission)

- admission_type_id
- discharge_disposition_id
- admission_source_id

## 4. Handle Missing Values

In [ ]:
fig = plot_missing_values(df)
save_figure(fig, '02_missing_before.png')
plt.show()

In [ ]:
# TODO: Choose strategy ('median' or 'mean') and categorical fill value.
df = handle_missing_values(df, strategy='median', categorical_fill='Unknown')
print(f'Missing values remaining: {df.isnull().sum().sum()}')

## 5. Outlier Treatment

In [ ]:
# TODO: List numeric columns where outlier removal is appropriate.
outlier_columns = []  # e.g. ['price', 'quantity']

if outlier_columns:
    df = remove_outliers_iqr(df, outlier_columns)

print(f'Shape after outlier removal: {df.shape}')

## 6. Feature Engineering

In [ ]:
# TODO: Add domain-specific derived features.
# Examples – uncomment and adapt as needed:

# df = add_log_transform(df, columns=['price'])
# df = add_interaction_term(df, 'col_a', 'col_b', name='col_a_x_col_b')
# df = add_binned_column(df, 'age', bins=4)

print('Feature engineering complete.')
print(f'Columns after engineering: {list(df.columns)}')

## 7. Encode Categorical Variables

In [ ]:
# TODO: List categorical columns to one-hot encode.
categorical_columns = []  # e.g. ['region', 'category']

if categorical_columns:
    df = encode_categoricals(df, categorical_columns)

print(f'Shape after encoding: {df.shape}')

## 8. Scale Numeric Features

In [ ]:
# TODO: Decide which numeric columns to scale and choose method.
# Note: exclude the target column from scaling.
target_column = 'target'  # <-- Change this
scale_cols = [
    c for c in df.select_dtypes(include=np.number).columns
    if c != target_column
]

# Comment out the lines below if you prefer to scale inside the modeling pipeline.
# df, scaler = scale_features(df, scale_cols, method='standard')

print('Scaling step ready – uncomment the line above to apply.')

## 9. Final Check & Save

In [ ]:
print(f'Final shape  : {df.shape}')
print(f'Dtypes:\n{df.dtypes.value_counts()}')
print(f'Missing total: {df.isnull().sum().sum()}')
df.head()

In [ ]:
save_processed_data(df, 'processed_dataset.csv')
print('Preprocessing complete. Proceed to Notebook 3 – Modeling.')